In [1]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, f1_score, confusion_matrix

df = pd.read_csv(r'C:\Users\USER\Desktop\5th sem\project1_daibatise\data\processed\mohammed_ajlan_p1w1_features.csv')


feature_cols = ["age","pregnancies","glucose","blood_pressure","skin_thickness","insulin","bmi","diabetes_pedigree","bmi_category","glucose_band"]
x= pd.get_dummies(df[feature_cols],drop_first=True)
y=df["outcome"]

x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [2]:
candidates = {
    "LogReg": (
        "scaled",
        LogisticRegression(max_iter=1000)
    ),
    "Decision Tree": (
        "raw",
        DecisionTreeClassifier(max_depth=3, random_state=42)
    ),
    "KNN": (
        "scaled",
        KNeighborsClassifier(n_neighbors=25)
    ),
    "SVC balanced": (
        "scaled",
        SVC(class_weight="balanced", random_state=42)
    ),
    "RF balanced": (
        "raw",
        RandomForestClassifier(
            max_depth=5,
            class_weight="balanced",
            random_state=42
        )
    )
}

TARGET_RECALL, GUARD_F1 = 0.70, 0.50  # Week 5 decision

In [3]:
print ('rows:',len(df),"| train:",len(x_train),"| test:", len(x_test))
print("diabetic in test set:", int(y_test.sum()))
print("featres after get dummies:",x.shape[1])
print("candidates:", list(candidates))
print("target: recall >=", TARGET_RECALL, "with F1 >=", GUARD_F1)

rows: 950 | train: 760 | test: 190
diabetic in test set: 55
featres after get dummies: 13
candidates: ['LogReg', 'Decision Tree', 'KNN', 'SVC balanced', 'RF balanced']
target: recall >= 0.7 with F1 >= 0.5


In [4]:
rows = []

for rs in [0, 1, 7, 42, 99]:
    Xa, Xb, ya, yb = train_test_split(
        x, y,
        test_size=0.2,
        random_state=rs,
        stratify=y
    )

    row = {"split": rs}

    for name, (kind, model) in candidates.items():
        m = clone(model)  

        if kind == "scaled":
            sc = StandardScaler().fit(Xa)
            m.fit(sc.transform(Xa), ya)
            pred = m.predict(sc.transform(Xb))

        else:
            m.fit(Xa, ya)
            pred = m.predict(Xb)
        row[name]=round(recall_score(yb,pred),3)
    rows.append(row)  

splits=pd.DataFrame(rows).set_index('split')
print('splits')  
spread=(splits.max() - splits.min()).round(3)
print('\nspread (best - worst):\n', spread.sort_values(ascending=False))

splits

spread (best - worst):
 Decision Tree    0.291
SVC balanced     0.145
RF balanced      0.091
LogReg           0.073
KNN              0.073
dtype: float64


# CROSS VALIDATION


K-Fold Cross-Validation is a technique used to check how well a machine learning model performs on different parts of a dataset.

In [5]:
from sklearn.model_selection import StratifiedKFold

cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for i,(tr,te) in enumerate(cv.split(x_train,y_train),1):
    print(f'fold{i}: train {len(tr)} score{len(te)}'
          f'diabetic in score fold{y_train.iloc[te].sum()}')

def make_model(kind, model):
    """Scaled models get the scaler INSIDE a Pipeline,so it refits per fold."""
    if kind == 'scaled':
        return Pipeline([("scale",StandardScaler()),('model',model)])
    return model

print()
for name,(kind,model)in candidates.items():
    print(f"{name:15s}->{type(make_model(kind,model)).__name__}")

fold1: train 608 score152diabetic in score fold43
fold2: train 608 score152diabetic in score fold43
fold3: train 608 score152diabetic in score fold44
fold4: train 608 score152diabetic in score fold44
fold5: train 608 score152diabetic in score fold44

LogReg         ->Pipeline
Decision Tree  ->DecisionTreeClassifier
KNN            ->Pipeline
SVC balanced   ->Pipeline
RF balanced    ->RandomForestClassifier


Setup cros validation the right way

Cross validate every candidate

In [6]:
from sklearn.model_selection import cross_validate

rows=[]
for name, (kind,model) in candidates.items():
    est=make_model(kind,model)
    res=cross_validate(est, x_train,y_train,cv=cv,
                       scoring=['recall','f1'])
    single=clone(est).fit(x_train,y_train)
    rows.append({
        'model':    name,
        'cv_recall': round(res["test_recall"].mean(),3),
        'cv_std':  round(res["test_recall"].std(),3),
        'cv_f1':   round(res["test_f1"].mean(),3),
        'week6_recall': round(recall_score(y_test,single.predict(x_test)),3),
        'folds':  np.round(res['test_recall'],3).tolist(),
    })

cv_table=pd.DataFrame(rows).sort_values('cv_recall',ascending=False)
print(cv_table.to_string(index=False))

        model  cv_recall  cv_std  cv_f1  week6_recall                               folds
 SVC balanced      0.721   0.046  0.559         0.618 [0.744, 0.791, 0.682, 0.727, 0.659]
  RF balanced      0.652   0.051  0.521         0.655 [0.721, 0.651, 0.682, 0.636, 0.568]
       LogReg      0.303   0.073  0.390         0.291  [0.419, 0.326, 0.318, 0.25, 0.205]
Decision Tree      0.216   0.076  0.296         0.291 [0.116, 0.326, 0.159, 0.205, 0.273]
          KNN      0.174   0.018  0.261         0.182 [0.186, 0.163, 0.159, 0.205, 0.159]


In [ ]:
cv_table["low"] = (cv_table["cv_recall"] - cv_table["cv_std"]).round(3)
cv_table["high"] = (cv_table["cv_recall"] + cv_table["cv_std"]).round(3)
cv_table["clears_target"] = cv_table["cv_recall"] >= TARGET_RECALL
cv_table["clears_guard"] = cv_table["cv_f1"] >= GUARD_F1
print(cv_table[["model","cv_recall","low","high","cv_f1",
   "clears_target","clears_guard"]].to_string(index=False))            


        model  cv_recall   low  high  cv_f1  clears_target  clears_guard
 SVC balanced      0.721 0.675 0.767  0.559           True          True
  RF balanced      0.652 0.601 0.703  0.521          False          True
       LogReg      0.303 0.230 0.376  0.390          False         False
Decision Tree      0.216 0.140 0.292  0.296          False         False
          KNN      0.174 0.156 0.192  0.261          False         False


## hyperparameter tuning

# hyperparameter 
A hyperparameter is a setting that we choose before training a Machine Learning model. It controls how the model learns from data.
# GridSearchCV
is a technique that tests all the specified combinations of hyperparameters to find the best combination using cross-validation.
# RandomizedSearchCV
is a technique that tests a selected number of randomly sampled hyperparameter combinations instead of testing every possible combination.

It also uses cross-validation to evaluate those combinations.

# Difference Between Parameter and Hyperparameter

A parameter is a value that a Machine Learning model learns automatically from the training data.

A hyperparameter is a setting chosen before training the model or selected through hyperparameter tuning.

In [10]:
rf=RandomForestClassifier(random_state=42)
for setting, Value in rf.get_params().items():
    print(f"{setting:25s}{Value}")

bootstrap                True
ccp_alpha                0.0
class_weight             None
criterion                gini
max_depth                None
max_features             sqrt
max_leaf_nodes           None
max_samples              None
min_impurity_decrease    0.0
min_samples_leaf         1
min_samples_split        2
min_weight_fraction_leaf 0.0
monotonic_cst            None
n_estimators             100
n_jobs                   None
oob_score                False
random_state             42
verbose                  0
warm_start               False


In [11]:
param_grid = {
    'max_depth': [3, 5, 8, None],
    'min_samples_leaf': [1, 5, 10],
    'class_weight': ['balanced', 'balanced_subsample']
}
combos = 1

for values in param_grid.values():
    combos *= len(values)

folds = cv.get_n_splits()
total_fits = combos * folds

print("Combinations:",combos)
print("cross-validation folds:", folds)
print("Total model fits:", total_fits)

Combinations: 24
cross-validation folds: 5
Total model fits: 120


run the grid search on the leading forest

In [13]:
from sklearn.model_selection import GridSearchCV

gs_recall = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=cv,
    scoring="recall",
    n_jobs=-1)


gs_recall.fit(x_train, y_train)

print("Best settings:", gs_recall.best_params_)
print("Best CV recall:", round(gs_recall.best_score_, 3))

top=pd.DataFrame(gs_recall.cv_results_)

print(
    top[["params","mean_test_score","std_test_score","rank_test_score"]]
    .sort_values("rank_test_score")
    .head(5)
    .to_string(index=False)
)

Best settings: {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 1}
Best CV recall: 0.739
                                                                        params  mean_test_score  std_test_score  rank_test_score
           {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 1}         0.738689        0.041181                1
           {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 5}         0.738584        0.041746                2
          {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 10}         0.734249        0.062548                3
{'class_weight': 'balanced_subsample', 'max_depth': 3, 'min_samples_leaf': 10}         0.720719        0.064830                4
 {'class_weight': 'balanced_subsample', 'max_depth': 3, 'min_samples_leaf': 5}         0.706871        0.065250                5
